# Man-in-the-Middle — Naive Data Injection to the GCS

A step past the stealth hijack of `15-mitm_hijack_stealth.ipynb`. There, the
attacker hid the *mode* change so the GCS kept seeing `AUTO`, but the real
position still flowed through — so the operator's **map** would show the drone
veering off course. Here the man-in-the-middle also **fabricates the position
feed**: the GCS's map shows the drone calmly flying its planned north mission
while the real vehicle is driven 100 m **west** by the hijack.

This is the drone-downlink form of a **false-data-injection / "ghost
telemetry"** attack — show the operator a normal picture while the physical
process is subverted (the same idea as the Stuxnet record-and-replay trick).

**Naive, by design** (proof of concept): the fake fix is dead-reckoned in a
straight line from the hijack point to the final planned waypoint at the cruise
speed. Only `GLOBAL_POSITION_INT` is faked — mission sequence, nav-controller
output and VFR_HUD still carry the truth, so a cross-check beyond the map would
expose it.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import GhostMissionStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin, layout, and mission

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0  # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 <- attacker hijacks here
home_wp = ENU(x=0, y=0, z=0)
north_100 = ENU(x=0, y=100, z=cruise_alt)
north_200 = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]
trigger_seq = 3

# Attacker's real destination: 100 m WEST, opposite the mission.
hijack_target = ENU(x=-100, y=0, z=cruise_alt)
print(f"Hijack target (ENU): {hijack_target}")

## Vehicle + ghost-mission attack

In [ ]:
mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_folder / "mitm_ghost.waypoints"),
    navigation_speed=speed,
    firmware=model.firmware,
)

# Same hijack as notebook 15 (switch to GUIDED at seq=3, fly 100 m west, hold),
# but GhostMissionStrategy ALSO rewrites the GCS's GLOBAL_POSITION_INT so the
# map keeps showing the planned north mission advancing. The stealth mode-mask
# is inherited, so mode AND position both look normal to the operator.
attack = GhostMissionStrategy(
    intervention=Intervention(
        trigger=MissionTrigger(seq=trigger_seq),
        plan=InterventionPlan.from_relative_path(
            relative_path=[hijack_target],
            enu_origin=enu_origin,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    ),
    route=mission_wps,
    navigation_speed=speed,
)

vehicle = SimVehicle.from_relative(
    model=model,
    sysid=sysid,
    color=Color.BLUE,
    plan=plan,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    mitm=attack,
)

# record_positions=True so we can replay the GCS's (fabricated) view afterwards.
gcs = SimGCS(name=f"BLUE_{Color.BLUE.emoji}", record_positions=True)
gcs.own(vehicle)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
target_marker = GazMarker(
    name="hijack_target",
    group="targets",
    pos=hijack_target,
    color=Color.RED,
)
gaz.markers.append(target_marker)
gaz.markers.append(origin_marker)

## Oracle

In [ ]:
orac = Oracle()
orac.add_vehicle(vehicle)

## Simulator + ghost-mission injection

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=1,
    terminals=[SimProcess.MITM, SimProcess.LOGIC, SimProcess.GCS],
    speedup=1,
)
simulator.preview()

In [ ]:
simulator.run(timeout=100)

## What the operator sees

The GCS's recorded track — the fabricated one. It follows the planned north
mission the whole time; nothing on the map hints at the hijack.

In [ ]:
orac.plot_trajectories(oracle=False, gcss=[gcs.name], waypoints=True);

## The reveal

Overlay the Oracle's own track (the **real** position, taken straight from the
vehicle's Remote ID feed, which the man-in-the-middle never touches). The real
drone breaks off **west** at the hijack, while the GCS's track keeps marching
**north** along the plan — the two diverge exactly at `trigger_seq`.

In [ ]:
orac.plot_trajectories(oracle=True, waypoints=True);